# NullWasted: Software Verification & Testing Suite
## Rekayasa Perangkat Lunak (RPL-2) — Kelompok 16 FTUI

Dokumen ini memuat eksekusi kasus uji terstruktur untuk modul inti backend NullWasted.
Setiap sel secara transparan menampilkan:
- **Injected Input**: Data mentah, parameter, atau HTTP payload yang disuntikkan ke dalam sistem.
- **System Logic**: Logika bisnis atau algoritma rekayasa yang dipicu.
- **Verification / Assertion**: Evaluasi kepatuhan luaran sistem terhadap kriteria akseptansi formal.

Standar Acuan: IEEE SWEBOK v4, ISO/IEC/IEEE 29119, ISO/IEC 25010, PerBPOM No. 16/2019.

### 0. Inisialisasi Environment & Konfigurasi Pengujian

In [ ]:
import sys
from pathlib import Path

backend_dir = Path.cwd().resolve()
if str(backend_dir) not in sys.path:
    sys.path.insert(0, str(backend_dir))

import pytest
from fastapi.testclient import TestClient
from sqlmodel import Session, SQLModel, create_engine
from sqlmodel.pool import StaticPool

from app.main import app
from app.core.database import get_session
from app.services.pricing import calculate_dynamic_pricing
from app.services.guardrail import evaluate_food_safety
from app.services.vector_store import vector_store

print("[INITIALIZATION] Backend modules and services loaded.")
print(f"[INJECTED CONFIG] Root Workspace : {backend_dir}")
print(f"[KNOWLEDGE BASE] Food Archetypes: {len(vector_store.items)} items (PerBPOM No. 16/2019).")
print("[VECTOR ENGINE] Embedding Cache : 3072-dimensional vector space.")
print("[DATABASE STATE] Target Database : In-Memory SQLite Isolation (Zero Persistent Contamination).")

---
### 1. Automated Regression Suite (Pytest Runner)
- **Injected Target**: Berkas uji `tests/test_unit_core.py` (10 kasus uji otomatis).
- **System Logic**: Menjalankan seluruh pengujian fungsional unit, batas diskon, aturan BPOM, dan integrasi API.
- **Assertion**: Nilai exit code wajib 0 (100% test cases lolos).

In [ ]:
import concurrent.futures

print("[RUNNER] Invoking Pytest on test suite: tests/test_unit_core.py\n")

def _run_pytest():
    return pytest.main(["-v", "tests/test_unit_core.py"])

# Run in an isolated worker thread so Pytest has a clean execution context independent of Jupyter's active event loop
with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
    exit_code = executor.submit(_run_pytest).result()

print(f"\n[SUITE RESULT] Execution finished with exit code: {int(exit_code)} (0 = 100% PASSED)")

---
### 2. Unit Testing: Dynamic Pricing Engine
- **Metode Rekayasa**: Boundary Value Analysis (BVA).
- **System Logic**: Formula time-decay yang memperhitungkan rasio waktu operasional tersisa terhadap jam tutup gerai.
- **Injected Inputs**:
  1. `original_price` (Harga dasar hidangan dalam Rupiah).
  2. `minutes_until_closing` (Sisa menit menuju jam tutup operasional gerai: 240 mnt, 120 mnt, 15 mnt).
  3. `custom_discount_percent` (Diskon manual di luar batas: 85% dan 30%).
- **Assertion**: Diskon wajib terkunci pada interval [50%, 70%] dan harga dibulatkan ke ratusan terdekat.

In [ ]:
pricing_scenarios = [
    {
        "case_id": "BVA-01",
        "desc": "Far from closing window (240 min left)",
        "injected_price": 20000,
        "injected_minutes": 240,
        "injected_custom_discount": None,
        "expected_discount": 50,
        "expected_price": 10000,
    },
    {
        "case_id": "BVA-02",
        "desc": "Preset Pill 1 standard window (120 min left)",
        "injected_price": 20000,
        "injected_minutes": 120,
        "injected_custom_discount": None,
        "expected_discount": 55,
        "expected_price": 9000,
    },
    {
        "case_id": "BVA-03",
        "desc": "Preset Pill 3 critical close window (15 min left)",
        "injected_price": 30000,
        "injected_minutes": 15,
        "injected_custom_discount": None,
        "expected_discount": 70,
        "expected_price": 9000,
    },
    {
        "case_id": "BVA-04",
        "desc": "Custom discount exceeds maximum ceiling (85% input)",
        "injected_price": 10000,
        "injected_minutes": 60,
        "injected_custom_discount": 85,
        "expected_discount": 70,
        "expected_price": 3000,
    },
    {
        "case_id": "BVA-05",
        "desc": "Custom discount below minimum floor (30% input)",
        "injected_price": 10000,
        "injected_minutes": 60,
        "injected_custom_discount": 30,
        "expected_discount": 50,
        "expected_price": 5000,
    },
    {
        "case_id": "BVA-06",
        "desc": "Odd nominal rounding verification (Rp16.250 / 160 min)",
        "injected_price": 16250,
        "injected_minutes": 160,
        "injected_custom_discount": None,
        "expected_discount": 50,
        "expected_price": 8100,
    },
]

for sc in pricing_scenarios:
    res = calculate_dynamic_pricing(
        original_price=sc["injected_price"],
        minutes_until_closing=sc["injected_minutes"],
        custom_discount_percent=sc["injected_custom_discount"],
    )
    passed = (res["discount_percent"] == sc["expected_discount"]) and (res["discounted_price"] == sc["expected_price"])
    status_str = "[PASS]" if passed else "[FAIL]"
    
    print(f"--- {sc['case_id']}: {sc['desc']} ---")
    print(f"  [INJECTED INPUT] Price: Rp{sc['injected_price']:,} | Remaining: {sc['injected_minutes']} min | Custom Discount: {sc['injected_custom_discount']}%")
    print(f"  [SYSTEM OUTPUT]  Calculated Discount: {res['discount_percent']}% | Final Price: Rp{res['discounted_price']:,}")
    print(f"  [VERIFICATION]   Expected: {sc['expected_discount']}% (Rp{sc['expected_price']:,}) -> {status_str}\n")

---
### 3. Unit Testing: Food Safety Guardrail
- **Metode Rekayasa**: Equivalence Partitioning berbasis PerBPOM No. 16/2019.
- **System Logic**: Menentukan ketahanan mikrobiologis pada suhu ruang berdasarkan komposisi bahan hidangan.
- **Injected Inputs**:
  1. `food_name` & `raw_text` (Nama makanan dan teks deskripsi bahan).
  2. `hours_since_cooked` (Durasi penyimpanan pada suhu ruang sejak selesai dimasak).
- **Assertion**:
  - High Perishability (santan/susu/krim) $> 4$ jam $\to$ Status `REJECTED`.
  - Medium Perishability (makanan matang siap saji) $\le 8$ jam $\to$ Status `SAFE`.
  - Low Perishability (produk bakery) $\le 24$ jam $\to$ Status `SAFE`.

In [ ]:
guardrail_scenarios = [
    {
        "case_id": "BPOM-01",
        "food_name": "Gulai Ayam Padang",
        "raw_text": "Ada sisa 3 porsi Gulai Ayam Padang santan kental, sudah 7 jam suhu ruang",
        "injected_hours": 7.0,
        "expected_status": "REJECTED",
        "expected_limit": 4.0,
    },
    {
        "case_id": "BPOM-02",
        "food_name": "Spageti Carbonara",
        "raw_text": "Spageti carbonara saus krim keju dan kuning telur matang 6 jam lalu",
        "injected_hours": 6.0,
        "expected_status": "REJECTED",
        "expected_limit": 4.0,
    },
    {
        "case_id": "BPOM-03",
        "food_name": "Caesar Salad with Mayo",
        "raw_text": "Salad sayur dengan saus mayones dan telur rebus dibuat 5 jam lalu",
        "injected_hours": 5.0,
        "expected_status": "REJECTED",
        "expected_limit": 4.0,
    },
    {
        "case_id": "BPOM-04",
        "food_name": "Nasi Bento Ayam",
        "raw_text": "Ada 5 porsi Nasi Bento Ayam baru matang 2 jam lalu",
        "injected_hours": 2.0,
        "expected_status": "SAFE",
        "expected_limit": 8.0,
    },
    {
        "case_id": "BPOM-05",
        "food_name": "Croissant Cokelat",
        "raw_text": "Sisa 4 porsi Croissant Cokelat baru matang 1.5 jam lalu",
        "injected_hours": 1.5,
        "expected_status": "SAFE",
        "expected_limit": 24.0,
    },
    {
        "case_id": "BPOM-06",
        "food_name": "Pizza Meat Lovers",
        "raw_text": "3 loyang pizza panggang matang 3 jam lalu",
        "injected_hours": 3.0,
        "expected_status": "SAFE",
        "expected_limit": 8.0,
    },
]

for g in guardrail_scenarios:
    eval_res = evaluate_food_safety(food_name=g["food_name"], raw_text=g["raw_text"], hours_since_cooked=g["injected_hours"])
    status_actual = eval_res["safety_status"]
    limit_actual = eval_res["bpom_max_hours"]
    passed = (status_actual == g["expected_status"]) and (limit_actual == g["expected_limit"])
    verdict = "[PASS]" if passed else "[FAIL]"
    
    print(f"--- {g['case_id']}: {g['food_name']} ---")
    print(f"  [INJECTED INPUT] Raw Text: \"{g['raw_text']}\"")
    print(f"  [INJECTED INPUT] Stored Duration: {g['injected_hours']} hours")
    print(f"  [SYSTEM OUTPUT]  Category: {eval_res['category']} | BPOM Limit: {limit_actual} hours | Decision: {status_actual}")
    print(f"  [REASON LOG]     {eval_res['safety_reason']}")
    print(f"  [VERIFICATION]   Expected: {g['expected_status']} (Limit: {g['expected_limit']}h) -> {verdict}\n")

---
### 4. Unit Testing: Semantic Vector Store Alignment
- **Metode Rekayasa**: Cosine Similarity Alignment pada ruang vektor 3072-dimensi.
- **System Logic**: Menghitung dot product vektor masukan terhadap 70 basis data kuliner terstandarisasi BPOM.
- **Injected Inputs**: String kueri kuliner informal/non-standar pedagang.
- **Assertion**: Skor kemiripan kosinus terhadap hidangan kanonikal terkait wajib $\ge 0.80$ dan kategori risiko BPOM sesuai.

In [ ]:
vector_queries = [
    {
        "query": "soto tangkar kuah santan gurih kental",
        "expected_canonical": "Soto Betawi Kuah Santan Susu",
        "expected_category": "HIGH_PERISHABILITY",
        "min_score": 0.80,
    },
    {
        "query": "creamy fettuccine carbonara extra cheese",
        "expected_canonical": "Spageti Carbonara Saus Krim Telur",
        "expected_category": "HIGH_PERISHABILITY",
        "min_score": 0.80,
    },
    {
        "query": "roti sobek artisan isi cokelat lumer panggang",
        "expected_canonical": "Roti Sobek Manis Cokelat Keju",
        "expected_category": "LOW_PERISHABILITY",
        "min_score": 0.80,
    },
    {
        "query": "pepperoni cheesy pizza slice hangat",
        "expected_canonical": "Pizza Daging & Keju Panggang",
        "expected_category": "MEDIUM_PERISHABILITY",
        "min_score": 0.80,
    },
]

for v in vector_queries:
    match, score = vector_store.search_similar_dish(v["query"])
    passed = (
        (score >= v["min_score"])
        and (match["canonical_name"] == v["expected_canonical"])
        and (match["category"] == v["expected_category"])
    )
    verdict = "[PASS]" if passed else "[FAIL]"
    
    print(f"[INJECTED QUERY]   : \"{v['query']}\"")
    print(f"[SYSTEM MATCH]     : Canonical: {match['canonical_name']} | Category: {match['category']} | Max Hours: {match['bpom_max_hours']}h")
    print(f"[SIMILARITY SCORE] : {score:.4f} (Threshold: >= {v['min_score']})")
    print(f"[VERIFICATION]     : Expected Target: {v['expected_canonical']} | {v['expected_category']} -> {verdict}\n")

---
### 5. Integration Testing: Full Closed-Loop API Lifecycle
- **Metode Rekayasa**: State Machine Integration Testing via `FastAPI TestClient`.
- **System Logic**: Menjalankan siklus hidup transaksi lengkap pada lingkungan SQLite in-memory murni.
- **Injected Payloads per Step**:
  1. `GET /health` (Pemeriksaan ketersediaan layanan).
  2. `POST /api/v1/listings/draft` (Injeksi teks listing raw pedagang).
  3. `POST /api/v1/listings/publish` (Injeksi hidangan santan basi untuk memicu blokir HTTP 400).
  4. `POST /api/v1/listings/publish` (Injeksi hidangan aman untuk memicu persistensi HTTP 200).
  5. `POST /api/v1/orders/reserve` (Injeksi reservasi pembeli untuk pemotongan stok atomik).
  6. `POST /api/v1/orders/complete-pickup` (Injeksi kode tiket kasir untuk status `COMPLETED`).
  7. `POST /api/v1/orders/complete-pickup` (Injeksi ulang kode tiket sama untuk membuktikan pencegahan *double redemption*).

In [ ]:
test_engine = create_engine("sqlite:///:memory:", connect_args={"check_same_thread": False}, poolclass=StaticPool)
SQLModel.metadata.create_all(test_engine)

def override_session():
    with Session(test_engine) as session:
        yield session

app.dependency_overrides[get_session] = override_session
client = TestClient(app)

print("[LIFECYCLE TEST] Starting 7-step transactional state-machine test...\n")

# Step 1
r1 = client.get("/health")
assert r1.status_code == 200
print("Step 1: Health Check Endpoint")
print("  [INJECTED REQUEST] GET /health")
print(f"  [SYSTEM RESPONSE]  HTTP {r1.status_code} | Body: {r1.json()}")
print("  [ASSERTION]        status == 'ok' -> [PASS]\n")

# Step 2
draft_payload = {"text": "Sisa 4 porsi Croissant Cokelat Rp20.000 matang 1 jam lalu tutup 2 jam lagi"}
r2 = client.post("/api/v1/listings/draft", json=draft_payload)
assert r2.status_code == 200
d = r2.json()
assert d["safety_status"] == "SAFE"
print("Step 2: Natural Language Listing Draft Extraction")
print(f"  [INJECTED PAYLOAD] POST /api/v1/listings/draft -> {draft_payload}")
print(f"  [SYSTEM RESPONSE]  Extracted: '{d['food_name']}' | Qty: {d['quantity']} | Disc: {d['recommended_discount_percent']}% | Safety: {d['safety_status']}")
print("  [ASSERTION]        safety_status == 'SAFE' -> [PASS]\n")

# Step 3
reject_payload = {
    "food_name": "Gulai Ayam Basi",
    "matched_canonical_dish": "Gulai Ayam",
    "category": "HIGH_PERISHABILITY",
    "quantity": 3,
    "original_price": 25000,
    "discount_percent": 50,
    "hours_since_cooked": 7.0,
    "bpom_max_hours": 4.0,
    "minutes_until_closing": 120,
    "safety_status": "REJECTED",
    "safety_reason": "Waktu simpan suhu ruang melampaui batas BPOM",
    "hazard_description": "Pertumbuhan toksin Bacillus cereus",
}
r3 = client.post("/api/v1/listings/publish", json=reject_payload)
assert r3.status_code == 400
print("Step 3: Food Safety Guardrail Enforcement (Reject Publication)")
print(f"  [INJECTED PAYLOAD] POST /api/v1/listings/publish -> Gulai Ayam Basi (7.0h > 4.0h BPOM Limit)")
print(f"  [SYSTEM RESPONSE]  HTTP {r3.status_code} Bad Request | Detail: {r3.json()['detail'][:50]}...")
print("  [ASSERTION]        HTTP Status == 400 (Expected Guardrail Block) -> [PASS]\n")

# Step 4
safe_payload = {
    "food_name": d["food_name"],
    "matched_canonical_dish": d["matched_canonical_dish"],
    "category": d["category"],
    "quantity": d["quantity"],
    "original_price": d["original_price"],
    "discount_percent": d["recommended_discount_percent"],
    "hours_since_cooked": d["hours_since_cooked"],
    "bpom_max_hours": d["bpom_max_hours"],
    "minutes_until_closing": d["minutes_until_closing"],
    "safety_status": d["safety_status"],
    "safety_reason": d["safety_reason"],
    "hazard_description": d["hazard_description"],
}
r4 = client.post("/api/v1/listings/publish", json=safe_payload)
assert r4.status_code == 200
pub_data = r4.json()
listing_id = pub_data["id"]
print("Step 4: Safe Listing Publication to Active Catalog")
print(f"  [INJECTED PAYLOAD] POST /api/v1/listings/publish -> Croissant Cokelat (1.5h <= 24.0h limit)")
print(f"  [SYSTEM RESPONSE]  HTTP {r4.status_code} OK | Listing ID: {listing_id} | Final Stock: {pub_data['quantity']}")
print("  [ASSERTION]        listing_id.startswith('lst-') -> [PASS]\n")

# Step 5
reserve_payload = {"listing_id": listing_id, "quantity": 1, "customer_name": "Mahasiswa FTUI"}
r5 = client.post("/api/v1/orders/reserve", json=reserve_payload)
assert r5.status_code == 200
order_data = r5.json()
order_code = order_data["order_code"]
print("Step 5: Order Reservation & Atomic Stock Decrement")
print(f"  [INJECTED PAYLOAD] POST /api/v1/orders/reserve -> {reserve_payload}")
print(f"  [SYSTEM RESPONSE]  HTTP {r5.status_code} OK | Issued Ticket: {order_code} | QR Payload: {order_data['qr_payload'][:30]}...")

# Verify stock decremented to 3
r_stock = client.get("/api/v1/listings/active")
updated_stock = next(item["quantity"] for item in r_stock.json() if item["id"] == listing_id)
assert updated_stock == 3
print(f"  [DATABASE CHECK]   Listing stock atomically decremented from 4 to {updated_stock}")
print("  [ASSERTION]        stock == 3 and status == 'RESERVED' -> [PASS]\n")

# Step 6
cashier_payload = {"order_code": order_code}
r6 = client.post("/api/v1/orders/complete-pickup", json=cashier_payload)
assert r6.status_code == 200
assert r6.json()["status"] == "COMPLETED"
print("Step 6: Cashier Verification & Order Redemption")
print(f"  [INJECTED PAYLOAD] POST /api/v1/orders/complete-pickup -> {cashier_payload}")
print(f"  [SYSTEM RESPONSE]  HTTP {r6.status_code} OK | Ticket Status: {r6.json()['status']} | Completed At: {r6.json()['completed_at']}")
print("  [ASSERTION]        status == 'COMPLETED' -> [PASS]\n")

# Step 7
r7 = client.post("/api/v1/orders/complete-pickup", json=cashier_payload)
assert r7.status_code == 400
print("Step 7: Anti-Double Redemption Security Enforcement")
print(f"  [INJECTED PAYLOAD] POST /api/v1/orders/complete-pickup -> Duplicate Scan of {order_code}")
print(f"  [SYSTEM RESPONSE]  HTTP {r7.status_code} Bad Request | Detail: {r7.json()['detail']}")
print("  [ASSERTION]        HTTP Status == 400 (Duplicate Redemption Prevented) -> [PASS]\n")

app.dependency_overrides.clear()
print("[LIFECYCLE RESULT] All 7 transactional steps executed and verified successfully.")

---
### 6. Matriks Ringkasan Kualitas Pengujian

| Tingkat Pengujian | Metode Rekayasa | Komponen yang Diverifikasi | Injected Input Utama | Kriteria Akseptansi |
| :--- | :--- | :--- | :--- | :--- |
| **Unit Testing** | Boundary Value Analysis | `calculate_dynamic_pricing` | Harga dasar, sisa menit (15, 120, 240), diskon kustom | Diskon terikat 50% - 70%, pembulatan ratusan rupiah |
| **Unit Testing** | Equivalence Partitioning | `evaluate_food_safety` | Teks bahan, durasi suhu ruang (1.5h s.d. 7.0h) | Kepatuhan PerBPOM No. 16/2019 (High 4h, Med 8h, Low 24h) |
| **Unit Testing** | Vector Similarity Alignment | `vector_store` | Kueri deskriptif informal (misal: soto tangkar, carbonara) | Kemiripan kosinus $\ge 0.80$ terhadap korpus 70 hidangan |
| **Integration Testing** | State Machine & In-Memory DB | `FastAPI TestClient` | Rangkaian payload HTTP (Draft, Publish, Reserve, Pickup) | Rantai status: DRAFT $\to$ PUBLISHED $\to$ RESERVED $\to$ COMPLETED |
| **Security Testing** | Negative Path Testing | `complete-pickup` | Payload duplikat tiket yang telah COMPLETED | HTTP 400 Bad Request (Pencegahan penukaran ganda) |